# Complete Ingestion Pipeline for Erica AI Tutor

This notebook combines all ingestion tasks:
1. **Connection Testing** - Verify MongoDB and Ollama
2. **Webpage Scraping** - Crawl course website
3. **Slide Extraction** - Extract text from linked PDFs and PPTX files
4. **Verification** - Show all ingested content and statistics

## Installation Requirements

Run this cell first to install required packages:

In [1]:
!pip install pymongo requests beautifulsoup4 PyPDF2 python-pptx lxml


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Import Libraries

In [2]:
import sys
sys.path.append('/workspace')

from pymongo import MongoClient
from datetime import datetime
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
import re
from typing import Set, List, Dict, Any
from collections import Counter

# PDF and PPTX extraction
import PyPDF2
from pptx import Presentation
import io

from ingestion.mongo_helper import MongoHelper

## 1. MongoDB Helper Class

Handles all database operations.

In [3]:
## Implemented in mongo_helper.py

## 2. Test Connections

Verify MongoDB and Ollama are accessible.

In [4]:
def test_mongodb():
    """Test MongoDB connection"""
    print("Testing MongoDB connection...")
    try:
        mongo = MongoHelper()
        counts = mongo.count_documents()
        print("MongoDB connected successfully!")
        print(f"  Current counts: {counts}")
        return mongo, True
    except Exception as e:
        print(f"MongoDB connection failed: {e}")
        return None, False

def test_ollama():
    """Test Ollama connection"""
    print("\nTesting Ollama connection...")
    try:
        response = requests.get("http://ollama:11434/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json()
            print("Ollama connected successfully!")
            print(f"  Available models: {[m['name'] for m in models.get('models', [])]}")
            return True
        else:
            print(f"Ollama returned status code: {response.status_code}")
            return False
    except Exception as e:
        print(f"Ollama connection failed: {e}")
        return False

# Run connection tests
mongo, mongodb_ok = test_mongodb()
ollama_ok = test_ollama()

if mongodb_ok and ollama_ok:
    print("\n" + "="*70)
    print("All systems ready!")
    print("="*70)
else:
    print("\n" + "="*70)
    print("Some systems failed - check configuration")
    print("="*70)

Testing MongoDB connection...
MongoDB connected successfully!
  Current counts: {'webpages': 185, 'slides': 0}

Testing Ollama connection...
Ollama connected successfully!
  Available models: ['qwen2.5:7b']

All systems ready!


## 3. Slide Extraction Functions

Extract text from PDF and PPTX files.

In [5]:
def extract_pdf_text(pdf_content: bytes) -> str:
    """Extract text from PDF bytes"""
    try:
        pdf_file = io.BytesIO(pdf_content)
        pdf_reader = PyPDF2.PdfReader(pdf_file)
        
        text_content = []
        for page in pdf_reader.pages:
            text_content.append(page.extract_text())
        
        return '\n'.join(text_content)
    except Exception as e:
        print(f"  Error extracting PDF text: {e}")
        return None

def extract_pptx_text(pptx_content: bytes) -> str:
    """Extract text from PPTX bytes"""
    try:
        pptx_file = io.BytesIO(pptx_content)
        presentation = Presentation(pptx_file)
        
        text_content = []
        for slide in presentation.slides:
            slide_text = []
            for shape in slide.shapes:
                if hasattr(shape, "text"):
                    slide_text.append(shape.text)
            text_content.append('\n'.join(slide_text))
        
        return '\n\n'.join(text_content)
    except Exception as e:
        print(f"  Error extracting PPTX text: {e}")
        return None

def find_slide_links_in_page(soup: BeautifulSoup, base_url: str) -> List[str]:
    """Find all PDF and PPTX links in a webpage"""
    slide_urls = set()
    
    for link in soup.find_all('a', href=True):
        href = link['href']
        full_url = urljoin(base_url, href)
        
        # Check if it's a PDF or PPTX
        if full_url.lower().endswith(('.pdf', '.pptx', '.ppt')):
            slide_urls.add(full_url)
    
    return list(slide_urls)

## 4. Complete Ingestion Pipeline

Main class that orchestrates webpage scraping and slide extraction.

In [6]:
class CompleteIngestion:
    def __init__(self, mongo: MongoHelper):
        self.mongo = mongo
        self.visited_urls: Set[str] = set()
        self.slide_files: Set[str] = set()
        self.base_url = "https://pantelis.github.io"
        
    def is_valid_url(self, url: str) -> bool:
        """Check if URL is valid and within course domain"""
        parsed = urlparse(url)
        
        # Must be from pantelis.github.io
        if 'pantelis.github.io' not in parsed.netloc:
            return False
        
        # Skip non-HTML content (we'll handle PDFs/PPTX separately)
        skip_extensions = ['.pdf', '.pptx', '.ppt', '.zip', '.jpg', '.png', '.gif', '.mp4']
        if any(url.lower().endswith(ext) for ext in skip_extensions):
            return False
            
        return True
    
    def extract_links(self, soup: BeautifulSoup, base_url: str) -> List[str]:
        """Extract all valid links from a page"""
        links = []
        for link in soup.find_all('a', href=True):
            url = urljoin(base_url, link['href'])
            url = url.split('#')[0]  # Remove fragments
            if self.is_valid_url(url) and url not in self.visited_urls:
                links.append(url)
        return links
    
    def ingest_webpage(self, url: str, max_depth: int = 3, current_depth: int = 0):
        """Recursively fetch and store webpage content, plus extract slides"""
        if url in self.visited_urls or current_depth > max_depth:
            return
        
        self.visited_urls.add(url)
        
        try:
            print(f"{'  ' * current_depth}Fetching: {url}")
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.text, 'html.parser')
            title = soup.title.string if soup.title else url
            
            # Find slide files (PDFs/PPTX) in this page
            slide_urls = find_slide_links_in_page(soup, url)
            for slide_url in slide_urls:
                if slide_url not in self.slide_files:
                    self.slide_files.add(slide_url)
                    print(f"{'  ' * current_depth}  Found slide: {slide_url.split('/')[-1]}")
            
            # Remove script, style, nav, footer
            for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
                tag.decompose()
            
            # Extract main content
            content = soup.get_text(separator='\n', strip=True)
            
            # Store webpage in MongoDB
            metadata = {
                'depth': current_depth,
                'word_count': len(content.split()),
                'slides_found': len(slide_urls)
            }
            
            self.mongo.insert_webpage(url, title, content, metadata)
            print(f"{'  ' * current_depth}Stored webpage: {title[:50]}...")
            
            # Extract and follow links
            if current_depth < max_depth:
                links = self.extract_links(soup, url)
                time.sleep(0.5)  # Be nice to the server
                
                for link in links:
                    self.ingest_webpage(link, max_depth, current_depth + 1)
                    
        except Exception as e:
            print(f"{'  ' * current_depth}Failed: {url} - {e}")
    
    def ingest_slides(self):
        """Download and extract text from all discovered slide files"""
        print("\n" + "="*70)
        print(f"EXTRACTING SLIDE CONTENT ({len(self.slide_files)} files)")
        print("="*70)
        
        if not self.slide_files:
            print("No slide files found.")
            return
        
        success_count = 0
        for slide_url in self.slide_files:
            filename = slide_url.split('/')[-1]
            print(f"\nProcessing: {filename}")
            
            try:
                response = requests.get(slide_url, timeout=30)
                response.raise_for_status()
                
                # Extract text based on file type
                if slide_url.lower().endswith('.pdf'):
                    content = extract_pdf_text(response.content)
                elif slide_url.lower().endswith(('.pptx', '.ppt')):
                    content = extract_pptx_text(response.content)
                else:
                    print(f"  Unsupported file type")
                    continue
                
                if content:
                    metadata = {
                        'word_count': len(content.split()),
                        'file_type': slide_url.split('.')[-1].lower()
                    }
                    
                    self.mongo.insert_slide(
                        url=slide_url,
                        filename=filename,
                        content=content,
                        metadata=metadata
                    )
                    print(f"  Content extracted ({metadata['word_count']} words)")
                    success_count += 1
                else:
                    print(f"  Could not extract content")
                    
            except Exception as e:
                print(f"  Failed: {e}")
        
        print(f"\nSuccessfully extracted {success_count}/{len(self.slide_files)} slide files")
    
    def run_complete_ingestion(self, start_urls: List[str], max_depth: int = 3):
        """Run the complete ingestion pipeline"""
        print("="*70)
        print("STARTING COMPLETE INGESTION PIPELINE")
        print("="*70)
        
        # Step 1: Scrape webpages (and discover slides)
        print("\n[STEP 1/2] Scraping webpages...")
        for url in start_urls:
            self.ingest_webpage(url, max_depth=max_depth)
        
        print(f"\nScraped {len(self.visited_urls)} webpages")
        print(f"Found {len(self.slide_files)} slide files")
        
        # Step 2: Extract slide content
        print("\n[STEP 2/2] Extracting slide content...")
        self.ingest_slides()
        
        print("\n" + "="*70)
        print("INGESTION COMPLETE!")
        print("="*70)

## 5. Run the Complete Ingestion

Set CLEAR_EXISTING_DATA = True to start fresh, or False to append to existing data.

In [7]:
# Configuration
CLEAR_EXISTING_DATA = True  # Set to True to clear database before ingestion

# Clear existing data if requested
if CLEAR_EXISTING_DATA:
    print("Clearing existing data...")
    mongo.clear_all()
    print()

# Create ingestion instance
ingestion = CompleteIngestion(mongo)

# Define starting URLs
start_urls = [
    "https://pantelis.github.io/",
    "https://pantelis.github.io/book/foundations/",
    "https://pantelis.github.io/book/dnn/",
    "https://pantelis.github.io/book/2d-perception/",
    "https://pantelis.github.io/book/kinematics/",
    "https://pantelis.github.io/book/state-estimation/",
    "https://pantelis.github.io/book/llm/",
    "https://pantelis.github.io/book/multimodal/", # Q2 CLIP is here
    "https://pantelis.github.io/book/task-planning/",
    "https://pantelis.github.io/book/global-planning/",
    "https://pantelis.github.io/book/local-planning/",
    "https://pantelis.github.io/book/mdp/",
    "https://pantelis.github.io/book/rl/",
    "https://pantelis.github.io/book/vla/",
    "https://pantelis.github.io/courses/cv/",
    "https://pantelis.github.io/courses/ai/",
    "https://pantelis.github.io/data-mining/intro.html",
    "https://pantelis.github.io/aiml-common/lectures/vae/elbo-optimization/elbo_optimization_torch.html",
    "https://pantelis.github.io/cs634/docs/",
    "https://pantelis.github.io/cs634/docs/common/lectures/vae/",
]

# Run complete ingestion
ingestion.run_complete_ingestion(start_urls, max_depth=3)

Clearing existing data...
All collections cleared!

STARTING COMPLETE INGESTION PIPELINE

[STEP 1/2] Scraping webpages...
Fetching: https://pantelis.github.io/
Stored webpage: Back2classroom – Engineering AI Agents...
  Fetching: https://pantelis.github.io/courses.html
  Stored webpage: Courses – Engineering AI Agents...
    Fetching: https://pantelis.github.io/ai
    Failed: https://pantelis.github.io/ai - 404 Client Error: Not Found for url: https://pantelis.github.io/ai
    Fetching: https://pantelis.github.io/robotics
    Failed: https://pantelis.github.io/robotics - 404 Client Error: Not Found for url: https://pantelis.github.io/robotics
    Fetching: https://pantelis.github.io/cv
    Failed: https://pantelis.github.io/cv - 404 Client Error: Not Found for url: https://pantelis.github.io/cv
  Fetching: https://pantelis.github.io/aiml-common/lectures/learning-problem/index.html
  Stored webpage: The Learning Problem – Engineering AI Agents...
  Fetching: https://pantelis.github.io/a

## 6. Verification & Statistics

Display comprehensive statistics about ingested content.

In [8]:
# Get document counts
counts = mongo.count_documents()

print("="*70)
print("INGESTION SUMMARY")
print("="*70)
print(f"Webpages:      {counts['webpages']:3d}")
print(f"Slides:        {counts['slides']:3d}")
print(f"TOTAL:         {sum(counts.values()):3d}")

# Calculate total words
total_words = 0
word_counts = []

# Webpage words
for doc in mongo.db.webpages.find({}, {'metadata.word_count': 1}):
    wc = doc.get('metadata', {}).get('word_count', 0)
    total_words += wc
    word_counts.append(wc)

# Slide words
for doc in mongo.db.slides.find({}, {'metadata.word_count': 1}):
    wc = doc.get('metadata', {}).get('word_count', 0)
    total_words += wc

print("\n" + "="*70)
print("CONTENT STATISTICS")
print("="*70)
print(f"Total words ingested:       {total_words:,}")
if counts['webpages'] > 0:
    print(f"Average words per webpage:  {total_words // max(sum(counts.values()), 1):,}")
    print(f"Min words per webpage:      {min(word_counts) if word_counts else 0:,}")
    print(f"Max words per webpage:      {max(word_counts) if word_counts else 0:,}")

INGESTION SUMMARY
Webpages:      185
Slides:         30
TOTAL:         215

CONTENT STATISTICS
Total words ingested:       1,405,075
Average words per webpage:  6,535
Min words per webpage:      6
Max words per webpage:      509,866


## 7. List All Ingested URLs

Required for M2 milestone - show all URLs that were ingested.

In [9]:
print("="*70)
print("ALL INGESTED URLs (M2 REQUIREMENT)")
print("="*70)

for doc_type, url in mongo.get_all_urls():
    print(f"[{doc_type:8s}] {url}")

ALL INGESTED URLs (M2 REQUIREMENT)
[webpage ] https://pantelis.github.io/
[webpage ] https://pantelis.github.io/courses.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/learning-problem/index.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/scene-understanding/scene-understanding-intro/index.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/scene-understanding/semantic-segmentation/maskrcnn/index.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/scene-understanding/semantic-segmentation/maskrcnn/tf/demo.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/scene-understanding/semantic-segmentation/maskrcnn/tf/inspect_data.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/scene-understanding/semantic-segmentation/maskrcnn/tf/inspect_model.html
[webpage ] https://pantelis.github.io/aiml-common/lectures/scene-understanding/semantic-segmentation/maskrcnn/tf/inspect_weights
[webpage ] https://pantelis.github.i

## 8. Sample Content from Each Type

In [10]:
print("="*70)
print("SAMPLE WEBPAGE")
print("="*70)
webpage = mongo.db.webpages.find_one()
if webpage:
    print(f"Title: {webpage['title']}")
    print(f"URL: {webpage['url']}")
    print(f"Word Count: {webpage['metadata'].get('word_count', 'N/A')}")
    print(f"\nFirst 300 chars:\n{webpage['content'][:300]}...\n")

print("="*70)
print("SAMPLE SLIDE CONTENT")
print("="*70)
slide = mongo.db.slides.find_one()
if slide:
    print(f"Filename: {slide['filename']}")
    print(f"URL: {slide.get('url', 'N/A')}")
    print(f"Word Count: {slide['metadata'].get('word_count', 'N/A')}")
    print(f"\nFirst 300 chars:\n{slide['content'][:300]}...\n")
else:
    print("No slides found.\n")

SAMPLE WEBPAGE
Title: Back2classroom – Engineering AI Agents
URL: https://pantelis.github.io/
Word Count: 2104

First 300 chars:
Back2classroom – Engineering AI Agents
Welcome !
Learn the concepts and engineer AI agents with real-time perceptive and language understanding abilities.
Use
Jupyter
notebooks to learn the concepts from scratch.
Simulate AI agents with egomotion using the Robotic Operating System (ROS2).
Build real...

SAMPLE SLIDE CONTENT
Filename: RLbook2020.pdf
URL: http://incompleteideas.net/book/RLbook2020.pdf
Word Count: 250156

First 300 chars:
ii

Adaptive Computation and Machine Learning
Francis Bach, series editor
A complete list of books published in the Adaptive Computation and Machine Learning
series appears at the back of this book.
Reinforcement Learning:
An Introduction
second edition
Richard S. Sutton and Andrew G. Barto
The MIT ...



## 9. Topic Distribution Analysis

In [11]:
# Extract topics from URLs
topics = []
for doc_type, url in mongo.get_all_urls():
    parts = url.split('/')
    for part in parts:
        if part and part not in ['https:', '', 'pantelis.github.io', 'aiml-common', 'lectures', 'index.html']:
            topics.append(part.replace('-', ' ').replace('_', ' '))

topic_counts = Counter(topics)

print("="*70)
print("TOP 20 TOPICS IN INGESTED CONTENT")
print("="*70)

for topic, count in topic_counts.most_common(20):
    print(f"{count:3d} - {topic}")

TOP 20 TOPICS IN INGESTED CONTENT
 28 - nlp
 25 - book
 24 - mdp
 21 - scene understanding
 15 - pdf
 13 - planning
 12 - arxiv.org
 11 - reinforcement learning
 11 - http:
 10 - semantic segmentation
  9 - maskrcnn
  9 - transformers
  9 - optimization
  8 - task planning
  8 - object detection
  8 - kinematics
  7 - language models
  7 - dynamic programming algorithms
  7 - mdp workshop
  6 - logical reasoning


## 10. Export Summary for M2 Submission

Generate a text file with all URLs for easy submission.

In [12]:
with open('/workspace/M2_ingested_urls.txt', 'w') as f:
    f.write("="*70 + "\n")
    f.write("M2 MILESTONE - ALL INGESTED URLs\n")
    f.write("="*70 + "\n\n")
    
    counts = mongo.count_documents()
    f.write(f"Total Webpages: {counts['webpages']}\n")
    f.write(f"Total Slides:   {counts['slides']}\n")
    f.write(f"TOTAL:          {sum(counts.values())}\n\n")
    
    f.write("="*70 + "\n")
    f.write("ALL URLS\n")
    f.write("="*70 + "\n\n")
    
    for doc_type, url in mongo.get_all_urls():
        f.write(f"[{doc_type:8s}] {url}\n")

print("Exported URL list to: /workspace/M2_ingested_urls.txt")

Exported URL list to: /workspace/M2_ingested_urls.txt


## Summary

This notebook provides a complete ingestion pipeline that:

1. Scrapes course webpages recursively
2. Discovers linked PDF and PPTX files
3. Extracts text content from slides
4. Stores everything in MongoDB with proper metadata
5. Provides comprehensive statistics and verification

Next Steps:
- Move on to M3: Knowledge Graph Construction
- Use the ingested content to extract concepts and build the pedagogical graph

In [13]:
## DEBUG
for doc in mongo.db.webpages.find():
    if 'jensen' in doc['content'].lower():
        print(f"Title: {doc['title']}")
        print(f"URL: {doc['url']}")
        # Check what's in that page about Jensen
        idx = doc['content'].lower().find('jensen')
        print(f"Context: {doc['content'][idx:idx+500]}\n")

Title: Variational Autoencoder from Scratch - Torch – Engineering AI Agents
URL: https://pantelis.github.io/aiml-common/lectures/vae/elbo-optimization/elbo_optimization_torch.html
Context: Jensen's Inequality
=: KL[q(Z|x) || p(x|Z)p(Z)]
= -E_{Z~q(Z|x)}[log p(x|Z)] + KL[q(Z|x) || p(Z)]
```
-or-
```none
-log p(x)
= KL[q(Z|x) || p(x|Z)p(Z)] - KL[q(Z|x) || p(Z|x)]
<= KL[q(Z|x) || p(x|Z)p(Z)                        # Positivity of KL
= -E_{Z~q(Z|x)}[log p(x|Z)] + KL[q(Z|x) || p(Z)]
```
The `-E_{Z~q(Z|x)}[log p(x|Z)]` term is an expected reconstruction loss and
`KL[q(Z|x) || p(Z)]` is a kind of distributional regularizer.
This implementation supports both standard normal prior and mixtur

